# DeiT-LT Scaling: ImageNet-LT

This notebook scales the baseline DeiT-LT model to the massive **ImageNet-LT** dataset (1000 classes). We will use a larger backbone (`DeiT-Small`) and replace the CIFAR-10 dataloaders with standard `torchvision` ImageNet-LT configurations.

## 1. Environment Setup


In [ ]:
!pip install timm==0.4.12 torch torchvision scikit-learn seaborn matplotlib scipy


## 2. Imports and Teacher Weights


In [ ]:
import os
import time
import math
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.scheduler import create_scheduler
from timm.optim import create_optimizer

# 1. Download Teacher Weights (IF 50)
teacher_url = "https://api.wandb.ai/artifactsV2/default/pradipto611/QXJ0aWZhY3Q6Nzk3NzA4NTEx/fc15814c0ce158e6987110b256248e18/paco_sam_ckpt_cf10_if50.pth.tar"
weight_path = "teacher_cifar10_lt_if50.pth"
if not os.path.exists(weight_path):
    print("Downloading Official ResNet-32 Teacher (IF=50)...")
    urllib.request.urlretrieve(teacher_url, weight_path)
    print("Downloaded!")



## 3. Dataset Generation (CIFAR-10 LT)


In [ ]:
# make_long_tail function removed for ImageNet-LT

## 4. Models (DeiT-Tiny Dual-Expert & ResNet-32 Teacher)


In [ ]:
# --- STUDENT: DeiT-Small for ImageNet-LT ---
import timm
import types
class DeiTLT(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.model = timm.create_model("deit_small_patch16_224", pretrained=False, num_classes=num_classes)
        self.embed_dim = self.model.embed_dim
        self.dist_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        
        # Override forward_features to include dist_token
        def new_forward_features(self_model, x):
            B = x.shape[0]
            x = self_model.patch_embed(x)
            cls_tokens = self_model.cls_token.expand(B, -1, -1)
            dist_tokens = self.dist_token.expand(B, -1, -1)
            x = torch.cat((cls_tokens, dist_tokens, x), dim=1)
            x = x + self_model.pos_embed
            x = self_model.pos_drop(x)
            for blk in self_model.blocks:
                x = blk(x)
            x = self_model.norm(x)
            return x[:, 0], x[:, 1] # Return both CLS and DIST
        
        self.model.forward_features = types.MethodType(new_forward_features, self.model)
        self.model.dist_token = self.dist_token
        self.model.pos_embed = nn.Parameter(torch.zeros(1, self.model.patch_embed.num_patches + 2, self.embed_dim))
        
        self.head_cls = nn.Linear(self.embed_dim, num_classes)
        self.head_dist = nn.Linear(self.embed_dim, num_classes)
        
    def forward(self, x, return_features=False):
        cls_tok, dist_tok = self.model.forward_features(x)
        logits_cls = self.head_cls(cls_tok)
        logits_dist = self.head_dist(dist_tok)
        if return_features:
            return logits_cls, logits_dist, cls_tok, dist_tok
        return logits_cls, logits_dist

model = DeiTLT(num_classes=1000)
print("DeiT-Small initialized for ImageNet-LT")


## 5. Main Training Pipeline


In [ ]:
import torch
# --- Configuration ---
class Config:
    epochs = 90
    num_classes = 1000
    batch_size = 128
    lr = 1e-3
    warmup_epochs = 5
    drw_epoch = int(epochs * 0.9)
    weight_decay = 0.05
    mixup = 0.8
    cutmix = 1.0
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

args = Config()
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

# --- ImageNet-LT Dataloaders ---
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os

IMAGENET_PATH = '/kaggle/input/imagenet/ILSVRC/Data/CLS-LOC/'

class ImageNetLT(Dataset):
    def __init__(self, root, txt_file, transform=None):
        self.img_path = []
        self.labels = []
        self.transform = transform
        if os.path.exists(txt_file):
            with open(txt_file) as f:
                for line in f:
                    self.img_path.append(os.path.join(root, line.split()[0]))
                    self.labels.append(int(line.split()[1]))
        else:
            print(f"WARNING: {txt_file} not found. Please upload to Kaggle.")
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        path = self.img_path[index]
        label = self.labels[index]
        try:
            with open(path, 'rb') as f:
                sample = Image.open(f).convert('RGB')
        except:
            sample = Image.new('RGB', (224, 224))
        if self.transform is not None:
            sample = self.transform(sample)
        return sample, label

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# train_dataset = ImageNetLT(IMAGENET_PATH, 'ImageNet_LT_train.txt', transform=transform_train)
# test_dataset = ImageNetLT(IMAGENET_PATH, 'ImageNet_LT_val.txt', transform=transform_test)
# train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=4)
# test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=4)

print("NOTE: Uncomment dataset instantiation after uploading ImageNet-LT txt splits.")


## 6. Training Metrics & Plots


In [ ]:
df_metrics = pd.DataFrame(metrics_history)
df_metrics.to_csv('training_metrics.csv', index=False)
display(df_metrics.tail())

plt.figure(figsize=(18,5))
plt.subplot(1, 3, 1)
plt.plot(df_metrics['epoch'], df_metrics['cls_acc'], label='CLS Acc')
plt.plot(df_metrics['epoch'], df_metrics['dist_acc'], label='DIST Acc')
plt.plot(df_metrics['epoch'], df_metrics['avg_acc'], label='AVG Acc', linewidth=2)
plt.axvline(x=args.drw_epoch, color='r', linestyle='--', label='DRW')
plt.legend()
plt.title('Token Accuracy over Training')

plt.subplot(1, 3, 2)
plt.plot(df_metrics['epoch'], df_metrics['cos_sim'], marker='o', color='purple')
plt.axvline(x=args.drw_epoch, color='r', linestyle='--')
plt.title('Cosine Similarity (CLS vs DIST)')

plt.subplot(1, 3, 3)
plt.plot(df_metrics['epoch'], df_metrics['teacher_entropy'], marker='^', color='orange')
plt.axvline(x=args.drw_epoch, color='r', linestyle='--')
plt.title('Teacher Entropy (OOD Distillation)')
plt.show()

# Confusion Matrices
cm_cls = confusion_matrix(all_test_lbls, all_test_preds_cls)
cm_dist = confusion_matrix(all_test_lbls, all_test_preds_dist)

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.heatmap(cm_cls, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('CLS Token Confusion Matrix (Epoch 300)')
plt.xlabel('Predicted')
plt.ylabel('True')

plt.subplot(1, 2, 2)
sns.heatmap(cm_dist, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('DIST Token Confusion Matrix (Epoch 300)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

